<a href="https://colab.research.google.com/github/pedromilken/Disciplinas-Doutorado/blob/main/aula0_3_seq2seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PGC305 - Tópicos especiais em LLM e Deep Learning - Prof. Marcelo Keese Albertini

# Preparação dos dados

Esta tarefa é inverter sequências de caracteres. Exemplo: **aabcd** em **dcbaa**.


In [ ]:
import torch
import torch.nn as nn
import random

chars = list("abcd ")
vocab = {ch: i for i, ch in enumerate(chars)} # Cada letra, ganha um número
inv_vocab = {i: ch for ch, i in vocab.items()}# Tabela de decodificação
vocab_size = len(vocab)

def encode(s): # Codifica letras em números
    return torch.tensor([vocab[c] for c in s], dtype=torch.long)

def decode(t): # Decodifica números em letras
    return ''.join(inv_vocab[int(x)] for x in t)

def random_seq(n=5): # Cria novas sequências
    return ''.join(random.choice(chars[:-1]) for _ in range(n))

# Gerar dados
pairs = [(encode(s), encode(s[::-1])) for s in [random_seq() for _ in range(50000)]]

max_len = max(len(x) for x, _ in pairs) # pega maior sequência

def pad(x):  # Preenche conjunto de dados em pad no último índice
    return torch.cat([x, torch.tensor([vocab[' ']] * (max_len - len(x)))], dim=0)

inputs = torch.stack([pad(x) for x, _ in pairs])
targets = torch.stack([pad(y) for _, y in pairs])

train_ds = torch.utils.data.TensorDataset(inputs, targets)
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=128, shuffle=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Veja um par

In [ ]:
print(pairs[1])

# Definição do modelo Seq2Seq com GRU

In [ ]:
import torch.nn as nn

# Generalized Encoder class to use either GRU or LSTM
class GenericEncoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size, rnn_type='GRU'):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        if rnn_type == 'GRU':
            self.rnn = nn.GRU(emb_size, hidden_size, batch_first=True)
        elif rnn_type == 'LSTM':
            self.rnn = nn.LSTM(emb_size, hidden_size, batch_first=True)
        else:
            raise ValueError("rnn_type must be 'GRU' or 'LSTM'")
        self.rnn_type = rnn_type

    def forward(self, x):
        x = self.embed(x)
        # output is (batch_size, seq_len, hidden_size)
        # hidden is (num_layers * num_directions, batch_size, hidden_size) for GRU
        # hidden is a tuple (h_n, c_n) for LSTM, where h_n and c_n are same shape as GRU hidden
        output, hidden = self.rnn(x)
        if self.rnn_type == 'LSTM':
            return hidden[0] # Return only h_n for LSTM, consistent with GRU's single output h_n
        return hidden # Returns h_n for GRU

# Generalized Decoder class to use either GRU or LSTM
class GenericDecoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size, rnn_type='GRU'):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        if rnn_type == 'GRU':
            self.rnn = nn.GRU(emb_size, hidden_size, batch_first=True)
        elif rnn_type == 'LSTM':
            self.rnn = nn.LSTM(emb_size, hidden_size, batch_first=True)
        else:
            raise ValueError("rnn_type must be 'GRU' or 'LSTM'")
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.rnn_type = rnn_type

    def forward(self, x, h_state):
        """
        x: tensor que indica a parte prévia correta
        h_state: tensor que indica o estado do encoder da parte prévia (always a single tensor h from GenericEncoder)
        """
        x = self.embed(x)

        if self.rnn_type == 'GRU':
            out, h_new = self.rnn(x, h_state)
        elif self.rnn_type == 'LSTM':
            # For LSTM, if h_state is just the hidden state, we create a cell state by duplicating it.
            # This makes the initial state (h_0, c_0) for LSTM, where h_0 = c_0 = h_state.
            out, (h_new, c_new) = self.rnn(x, (h_state, h_state))
            h_new = h_new # We return only the hidden state 'h' from LSTM for the next step or output
        else:
            raise ValueError("rnn_type must be 'GRU' or 'LSTM'")

        logits = self.fc(out)
        return logits, h_new # returns the hidden state 'h' for updating

# Seq2Seq class from the original prompt (unchanged, uses the provided encoder/decoder instances)
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt):
        h = self.encoder(src)
        # usa contexto correto anterior e estado atual para prever o tgt[:, -1]
        logits, _ = self.decoder(tgt[:, :-1], h)
        return logits

# --- Instantiation of the 4 possible scenarios for analysis ---
# Assuming vocab_size, emb_size, hidden_size are defined from previous cells.
# In this notebook, they are: vocab_size=5, emb_size=32, hidden_size=64

print(f"Definindo 4 modelos Seq2Seq com diferentes tipos de RNN para Encoder/Decoder (GRU/LSTM):")

# Scenario 1: Encoder GRU, Decoder GRU
encoder_gru_gru = GenericEncoder(vocab_size, emb_size, hidden_size, rnn_type='GRU')
decoder_gru_gru = GenericDecoder(vocab_size, emb_size, hidden_size, rnn_type='GRU')
model_gru_gru = Seq2Seq(encoder_gru_gru, decoder_gru_gru)
print(f"Modelo GRU-GRU: Encoder type={encoder_gru_gru.rnn_type}, Decoder type={decoder_gru_gru.rnn_type}")

# Scenario 2: Encoder GRU, Decoder LSTM
encoder_gru_lstm = GenericEncoder(vocab_size, emb_size, hidden_size, rnn_type='GRU')
decoder_gru_lstm = GenericDecoder(vocab_size, emb_size, hidden_size, rnn_type='LSTM')
model_gru_lstm = Seq2Seq(encoder_gru_lstm, decoder_gru_lstm)
print(f"Modelo GRU-LSTM: Encoder type={encoder_gru_lstm.rnn_type}, Decoder type={decoder_gru_lstm.rnn_type}")

# Scenario 3: Encoder LSTM, Decoder GRU
encoder_lstm_gru = GenericEncoder(vocab_size, emb_size, hidden_size, rnn_type='LSTM')
decoder_lstm_gru = GenericDecoder(vocab_size, emb_size, hidden_size, rnn_type='GRU')
model_lstm_gru = Seq2Seq(encoder_lstm_gru, decoder_lstm_gru)
print(f"Modelo LSTM-GRU: Encoder type={encoder_lstm_gru.rnn_type}, Decoder type={decoder_lstm_gru.rnn_type}")

# Scenario 4: Encoder LSTM, Decoder LSTM
encoder_lstm_lstm = GenericEncoder(vocab_size, emb_size, hidden_size, rnn_type='LSTM')
decoder_lstm_lstm = GenericDecoder(vocab_size, emb_size, hidden_size, rnn_type='LSTM')
model_lstm_lstm = Seq2Seq(encoder_lstm_lstm, decoder_lstm_lstm)
print(f"Modelo LSTM-LSTM: Encoder type={encoder_lstm_lstm.rnn_type}, Decoder type={decoder_lstm_lstm.rnn_type}")

print("\nPara uma análise mais aprofundada, você precisaria treinar e avaliar cada uma dessas instâncias de `model_...`.")


### Treinamento e Avaliação para os 4 Cenários de Modelo

Agora vamos treinar e testar cada uma das quatro combinações de modelos Encoder-Decoder (GRU/LSTM) para comparar seus desempenhos.

In [ ]:
# --- Training for all 4 models ---
models_to_train = {
    "GRU-GRU": model_gru_gru,
    "GRU-LSTM": model_gru_lstm,
    "LSTM-GRU": model_lstm_gru,
    "LSTM-LSTM": model_lstm_lstm,
}

epochs = 10 # Using the same number of epochs as the original training

print("\n--- Iniciando o treinamento para os 4 cenários ---")

for model_name, current_model in models_to_train.items():
    print(f"\nTreinando Modelo: {model_name}")
    current_model.to(device) # Ensure model is on the correct device

    # Re-initialize loss function and optimizer for each model
    loss_fn = nn.CrossEntropyLoss(ignore_index=vocab[' '])
    opt = torch.optim.Adam(current_model.parameters(), lr=1e-3)

    for epoch in range(epochs):
        current_model.train()
        total_loss = 0
        for xb, yb in train_dl:
            xb, yb = xb.to(device, dtype=torch.long), yb.to(device, dtype=torch.long)
            opt.zero_grad()
            logits = current_model(xb, yb)
            # Reshape logits and yb for CrossEntropyLoss
            loss = loss_fn(logits.reshape(-1, vocab_size), yb[:, 1:].reshape(-1))
            loss.backward()
            opt.step()
            total_loss += loss.item()
        print(f"  Epoch {epoch+1}: loss={total_loss/len(train_dl):.4f}")

print("\n--- Treinamento dos 4 cenários concluído ---")

# Vamos testar

### Teste de Predição para os 4 Modelos Treinados

Vamos gerar algumas predições para cada um dos modelos treinados para ver como se comportam.

In [ ]:
# --- Predictions for all 4 models ---
print("\n--- Iniciando predições para os 4 cenários ---")

test_sequences = [random_seq() for _ in range(5)] # Generate 5 random sequences for testing

for model_name, current_model in models_to_train.items():
    print(f"\nPredições para o Modelo: {model_name}")
    current_model.eval() # Set model to evaluation mode
    for s in test_sequences:
        # Ensure max_len is appropriate for the sequence length
        pred = predict(current_model, s, max_len=max_len) # Use the predict function
        print(f"  Entrada: {s} -> Saída Prevista: {pred}")

print("\n--- Predições dos 4 cenários concluídas ---")

# Exercício
Compare o resultado do uso do encoder de de sequências muito similares e muito diferentes. Por exemplo, codifique "aaaabb", "bbaaab", "cbcaccc" e "cccacbc" e depois faça uma figura das 2 componentes principais usando o método Principal Components Analysis (PCA) do pacote `sklearn.decomposition.PCA`.

### Análise PCA dos Embeddings do Encoder para os 4 Modelos

Vamos visualizar os embeddings do encoder para as mesmas sequências de exemplo usando PCA para cada um dos modelos treinados.

In [ ]:
# --- PCA visualization for all 4 models ---
print("\n--- Gerando visualizações PCA para os 4 cenários ---")

sequences_for_pca = [
    "aaaabb",
    "bbaaab",
    "cbcaccc",
    "cccacbc"
]

for model_name, current_model in models_to_train.items():
    print(f"\nGerando PCA para o Modelo: {model_name}")
    current_model.eval()
    embeddings = []
    with torch.no_grad():
        for s in sequences_for_pca:
            src = pad(encode(s)).unsqueeze(0).to(device, dtype=torch.long)
            h = current_model.encoder(src).squeeze().cpu().numpy()
            embeddings.append(h)

    embeddings_array = np.array(embeddings)

    pca = PCA(n_components=2)
    pca_results = pca.fit_transform(embeddings_array)

    plt.figure(figsize=(8, 6))
    plt.scatter(pca_results[:, 0], pca_results[:, 1])

    for i, seq in enumerate(sequences_for_pca):
        plt.annotate(seq, (pca_results[i, 0], pca_results[i, 1]), textcoords="offset points", xytext=(5,5), ha='center')

    plt.xlabel('Principal Component 1')
    plt.ylabel('Principal Component 2')
    plt.title(f'PCA of Encoder Embeddings for Sample Sequences ({model_name})')
    plt.grid(True)
    plt.show()

print("\n--- Visualizações PCA concluídas ---")

In [ ]:
models_to_train = {
    "GRU-GRU": model_gru_gru,
    "GRU-LSTM": model_gru_lstm,
    "LSTM-GRU": model_lstm_gru,
    "LSTM-LSTM": model_lstm_lstm,
}

def predict(model, s, max_len):
    model.eval()
    with torch.no_grad():
        # Encode the input sequence
        src = pad(encode(s)).unsqueeze(0).to(device, dtype=torch.long)
        # Get the encoder hidden state
        h = model.encoder(src)

        # Initialize target sequence with the start token (or first token for autoregressive)
        # For this task, we want to predict the reversed sequence, so we start with padding
        # and let the decoder generate tokens one by one.
        # Let's assume the decoder starts generating from the first output token.
        # The `tgt` input to the decoder should contain the tokens generated so far.

        # For autoregressive prediction, we need to feed the output of previous step as input to the next.
        # Start with an initial token (e.g., a special start-of-sequence token, or a padding token
        # if the model is designed to ignore it at the beginning).
        # Given the training setup `logits = self.decoder(tgt[:, :-1], h)`, the decoder
        # expects a sequence. We'll build the predicted sequence one character at a time.

        # Initialize the predicted sequence with a dummy starting token or a padding token
        # that gets ignored, or directly with the first predicted character if there's no BOS token.
        # Let's assume we initialize with the 'space' token and iteratively predict.
        predicted_tokens = [vocab[' ']] # Start with a padding token

        # The decoder loop
        for _ in range(max_len):
            # Prepare the current input for the decoder (last predicted token)
            decoder_input = torch.tensor([predicted_tokens[-1]], device=device, dtype=torch.long).unsqueeze(0)

            # For LSTM decoder, we need both h and c. If encoder is GRU, h_state is just h.
            # If decoder is LSTM and encoder is GRU/LSTM, the initial h_state is h from encoder.
            # The GenericDecoder expects (x, h_state). If it's LSTM, it duplicates h_state for c_state.
            # After the first step, the h_new returned by decoder should be used.

            # If it's the first step, use the encoder's output h. Otherwise, use the decoder's previous h_new.
            # Note: The `GenericDecoder`'s forward method returns `h_new` which is compatible for GRU and for LSTM (as h_n).
            # If the decoder is LSTM, `h` from encoder is used as both h_0 and c_0.
            # The `h` variable below needs to be updated by the decoder's `h_new`.

            logits, h = model.decoder(decoder_input, h)

            # Get the most likely next token
            next_token_id = logits.argmax(2).squeeze().item()
            predicted_tokens.append(next_token_id)

            # Stop if we predict a padding token (or end-of-sequence token)
            if next_token_id == vocab[' ']:
                break

        # Decode the predicted numerical sequence to characters
        # Exclude the initial dummy token if it was added for loop initialization
        return decode(torch.tensor(predicted_tokens[1:]))

def calculate_accuracy(model, data_loader):
    model.eval()
    correct_predictions = 0
    total_predictions = 0
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device, dtype=torch.long), targets.to(device, dtype=torch.long) # FIX: Added dtype=torch.long

            batch_size = inputs.size(0)
            # Initialize decoder input for the first step
            decoder_input = torch.tensor([vocab[' ']] * batch_size, device=device, dtype=torch.long).unsqueeze(1)
            # Get initial hidden state from encoder
            h = model.encoder(inputs)

            predicted_sequence = []
            for t in range(max_len): # max_len is the length of the longest sequence
                logits, h = model.decoder(decoder_input, h)
                next_token_ids = logits.argmax(2).squeeze(1) # Predict the next token for each item in batch
                predicted_sequence.append(next_token_ids)

                # Prepare for the next step: the predicted token becomes the decoder input
                decoder_input = next_token_ids.unsqueeze(1)

            predicted_sequence = torch.stack(predicted_sequence, dim=1)

            # Compare predicted sequence with target sequence (ignoring the first target token if it's start-of-sequence or padding)
            # For our task, the target sequence already includes padding, and we compare token by token.
            # We usually ignore padding tokens for accuracy calculation. Or we consider them if they align.
            # Let's count correct predictions for non-padding tokens in the target.

            # The target has max_len tokens (including padding). We compare predicted_sequence (also max_len, potentially).
            # Ensure predicted_sequence has the same length as targets (max_len)
            if predicted_sequence.shape[1] < max_len:
                # Pad predicted_sequence if it's shorter (e.g., stopped early due to ' ' prediction)
                padding_needed = max_len - predicted_sequence.shape[1]
                pad_tensor = torch.full((batch_size, padding_needed), vocab[' '], device=device, dtype=torch.long)
                predicted_sequence = torch.cat((predicted_sequence, pad_tensor), dim=1)

            # Calculate accuracy ignoring padding (' ') tokens in the target
            non_pad_mask = (targets != vocab[' '])
            correct_predictions += ((predicted_sequence == targets) * non_pad_mask).sum().item()
            total_predictions += non_pad_mask.sum().item()

    if total_predictions == 0:
        return 0.0
    return correct_predictions / total_predictions

print("\n--- Calculando a acurácia para os 4 cenários ---")

accuracies = {}
for model_name, current_model in models_to_train.items():
    print(f"\nAvaliando Modelo: {model_name}")
    current_model.to(device) # Ensure model is on the correct device
    accuracy = calculate_accuracy(current_model, train_dl) # Using train_dl for demonstration, ideally use a test_dl
    accuracies[model_name] = accuracy
    print(f"  Acurácia: {accuracy:.4f}")

print("\n--- Acurácia dos 4 cenários concluída ---")
print("\nResumo das Acurácias:")
for model_name, accuracy in accuracies.items():
    print(f"  {model_name}: {accuracy:.4f}")
